# Notebook_02: Preprocessing

### 🎯 Introduction
This notebook details the data preprocessing steps for the UCI Myocardial Infarction Complications dataset. The pipeline ensures data readiness by treating missing values, normalizing continuous variables, and performing one-hot encoding on categorical data. Crucially, the features are structured into three groups aligned with the 3-day hospitalization phases, and the 12 multi-class complication outputs are reduced to 5 core target outcomes for simplified predictive performance.

In [ ]:
# Core imports
import os
import numpy as np
import joblib
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- Global Configurations & Reproducibility ---
# Setting the random state for any randomized processing steps (if any)
RANDOM_STATE = 42

# Expanding pandas output display to view up to 50 columns during processing
pd.set_option("display.max_columns", 50)

print("✅ Preprocessing environment initialized successfully.")


✅ Preprocessing environment initialized successfully.


# Step 1: Load the DATA from EDA notebook

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load the raw train dataset while restoring the original patient index
train_df_raw = pd.read_csv('/content/drive/MyDrive/Mid-term Project/train_df_raw.csv', index_col=0)

print("Train raw dataframe shape:", train_df_raw.shape)
train_df_raw.head()


Train raw dataframe shape: (1360, 123)


,AGE,SEX,INF_ANAM,STENOK_AN,FK_STENOK,IBS_POST,IBS_NASL,GB,SIM_GIPERT,DLIT_AG,ZSN_A,nr_11,nr_01,nr_02,nr_03,nr_04,nr_07,nr_08,np_01,np_04,np_05,np_07,np_08,np_09,np_10,...,NA_R_1_n,NA_R_2_n,NA_R_3_n,NOT_NA_1_n,NOT_NA_2_n,NOT_NA_3_n,LID_S_n,B_BLOK_S_n,ANT_CA_S_n,GEPAR_S_n,ASP_S_n,TIKL_S_n,TRENT_S_n,FIBR_PREDS,PREDS_TAH,JELUD_TAH,FIBR_JELUD,A_V_BLOK,OTEK_LANC,RAZRIV,DRESSLER,ZSN,REC_IM,P_IM_STEN,LET_IS
0,77.0,1,2.0,1.0,1.0,2.0,NaN,3.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0
1,55.0,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0,0,0,0,0,0,0,0,0,0,0,0
2,52.0,1,0.0,0.0,0.0,2.0,NaN,2.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,3.0,2.0,2.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0
4,60.0,1,0.0,0.0,0.0,2.0,NaN,3.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0,0,0,0,0,0,0,0,0,0,0,0
6,70.0,1,1.0,1.0,2.0,1.0,NaN,2.0,0.0,7.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0,0,0,0,0,0,0,0,1,0,0,0


# Step 2: Check_and_remove_id column

In [ ]:
train_df = train_df_raw
def check_and_remove_id(df):
    """Function to check for the existence of an 'ID' column and remove it if present."""
    if "ID" in df.columns:
        print("🔍 'ID' column found in the dataset. Removing it now...")
        df = df.drop(columns=["ID"])
        print("✅ 'ID' column successfully removed.")
    else:
        print("ℹ️ 'ID' column does not exist in the dataset. No changes needed.")
    return df


# Executing the function on your df_train
train_df = check_and_remove_id(train_df)

ℹ️ 'ID' column does not exist in the dataset. No changes needed.


# STEP 3: Drop the high missing value columns (>60%)

In [ ]:
# List of specific columns to remove due to high missing values (>60%)
columns_to_remove = ["KFK_BLOOD", "IBS_NASL", "D_AD_KBRIG", "S_AD_KBRIG"]

# Dropping the columns from train_df
train_df = train_df.drop(columns=columns_to_remove, errors="ignore")

print("✅ High missing value columns (>60%) successfully removed:")
print(columns_to_remove)
print(f"New train_df shape: {train_df.shape}")

✅ High missing value columns (>60%) successfully removed:
['KFK_BLOOD', 'IBS_NASL', 'D_AD_KBRIG', 'S_AD_KBRIG']
New train_df shape: (1360, 119)


# Step 4: Separate features (X) and targets (Y)

In [ ]:
# Isolate targets from independent variables (features) while preserving the original index
X_train = train_df.iloc[:, :-12].copy()
y_train = train_df.iloc[:, -12:].copy()

print("\n✅ Step 4 Completed: Separated features (X) and targets (Y) with safe index alignment.")
print(f"X_train Shape: {X_train.shape} | y_train Shape: {y_train.shape}")



✅ Step 4 Completed: Separated features (X) and targets (Y) with safe index alignment.
X_train Shape: (1360, 107) | y_train Shape: (1360, 12)


# Step 5: Fixed target transformation with NAN preservation

In [ ]:
# Create the new DataFrame for the consolidated multi-label targets
y_train_meta = pd.DataFrame(index=y_train.index)

# --- Group 1: Severe Arrhythmias & Conduction Blocks ---
# FIBR_PREDS: Atrial Fibrillation
# FIBR_JELUD: Ventricular Fibrillation
# JELUD_TAH: Ventricular Tachycardia
# PREDS_TAH: Supraventricular Tachycardia
# A_V_BLOK: 3rd-degree AV Block
arrhythmia_cols = ['FIBR_PREDS', 'FIBR_JELUD', 'JELUD_TAH', 'PREDS_TAH', 'A_V_BLOK']
existing_arrhythmias = [c for c in arrhythmia_cols if c in y_train.columns]
y_train_meta['Severe_Arrhythmias_And_Blocks'] = y_train[existing_arrhythmias].max(axis=1, skipna=True)

# --- Group 2: Acute Pulmonary & Inflammatory Complications ---
# OTEK_LANC: Pulmonary Edema
# DRESSLER: Dressler Syndrome
acute_cols = ['OTEK_LANC', 'DRESSLER']
existing_acute = [c for c in acute_cols if c in y_train.columns]
y_train_meta['Acute_Pulmonary_And_Inflammatory'] = y_train[existing_acute].max(axis=1, skipna=True)

# --- Group 3: Chronic Heart Failure (StandAlone) ---
# ZSN: Chronic Heart Failure
if 'ZSN' in y_train.columns:
    y_train_meta['Chronic_Heart_Failure'] = y_train['ZSN']

# --- Group 4: Overall Mortality (Lethal Outcome) ---
# LET_IS: Lethal Outcome / Cause of Death (0 = Alive, >0 = Deceased)
if 'LET_IS' in y_train.columns:
    y_train_meta['Overall_Mortality'] = np.where(
        y_train['LET_IS'].isna(),
        np.nan,
        (y_train['LET_IS'] > 0).astype(float)
    )

# --- Group 5: Ischemic Relapse & Mechanical Ruptures ---
# REC_IM: Relapse of Myocardial Infarction
# P_IM_STEN: Post-infarction Angina
# RAZRIV: Myocardial Rupture
ischemic_cols = ['REC_IM', 'P_IM_STEN', 'RAZRIV']
existing_ischemic = [c for c in ischemic_cols if c in y_train.columns]
y_train_meta['Ischemic_Relapse_And_Myocardial_Rupture'] = y_train[existing_ischemic].max(axis=1, skipna=True)

print("✅ Step 5 Completed: Targets successfully restructured into 5 clear clinical groups.")
print(f"New Target Shape: {y_train_meta.shape}")

# Print class densities to monitor data imbalance
for col in y_train_meta.columns:
    print(f"\n--- Value counts for: {col} ---")
    print(y_train_meta[col].value_counts(dropna=True))
    print(f"Positive Class Density: {(y_train_meta[col].mean() * 100):.1f}%")


✅ Step 5 Completed: Targets successfully restructured into 5 clear clinical groups.
New Target Shape: (1360, 5)

--- Value counts for: Severe_Arrhythmias_And_Blocks ---
Severe_Arrhythmias_And_Blocks
0    1113
1     247
Name: count, dtype: int64
Positive Class Density: 18.2%

--- Value counts for: Acute_Pulmonary_And_Inflammatory ---
Acute_Pulmonary_And_Inflammatory
0    1179
1     181
Name: count, dtype: int64
Positive Class Density: 13.3%

--- Value counts for: Chronic_Heart_Failure ---
Chronic_Heart_Failure
0    1045
1     315
Name: count, dtype: int64
Positive Class Density: 23.2%

--- Value counts for: Overall_Mortality ---
Overall_Mortality
0.0    1143
1.0     217
Name: count, dtype: int64
Positive Class Density: 16.0%

--- Value counts for: Ischemic_Relapse_And_Myocardial_Rupture ---
Ischemic_Relapse_And_Myocardial_Rupture
0    1091
1     269
Name: count, dtype: int64
Positive Class Density: 19.8%


### 📊 Clinical Multi-Label Target Space Alignment Matrix

| Severe Arrhythmias & Blocks <br>(Group 1) | Acute Pulmonary & Inflammatory <br>(Group 2) | Chronic Heart Failure <br>(Group 3) | Overall Mortality <br>(Group 4) | Ischemic Relapse & Ruptures <br>(Group 5) |
| :--- | :--- | :--- | :--- | :--- |
| • `FIBR_PREDS` (Atrial Fib) | • `OTEK_LANC` (Pulmonary Edema) | • `ZSN` (Chronic Heart Failure) | • `LET_IS` (Cause of Death) | • `REC_IM` (Infarction Relapse) |
| • `FIBR_JELUD` (Ventricular Fib) | • `DRESSLER` (Dressler Syndrome) | | | • `P_IM_STEN` (Post-infarct Angina) |
| • `JELUD_TAH` (Ventricular Tach) | | | | • `RAZRIV` (Myocardial Rupture) |
| • `PREDS_TAH` (Supraventricular Tach) | | | | |
| • `A_V_BLOK` (3rd-degree AV Block) | | | | |


# Step 6: Sanity check: verifying the clincial target grouping logic

In [ ]:
print("=== VERIFYING GROUPING INTEGRITY (TRAIN SET ONLY) ===")

# Check 1: Verify the row count matches perfectly across features and new targets
assert len(X_train) == len(y_train_meta), "❌ Critical Error: Row mismatch between X and New Y!"
print(f"✅ Row Alignment Check Passed: Both X and New Y contain exactly {len(y_train_meta)} rows.")

# Check 2: Verify the logical grouping of Severe Arrhythmias & Conduction Blocks (Including A_V_BLOK)
# Note: Using min_count=1 logic simulation in .any() or treating NaNs carefully
original_arrhythmia_sum = y_train[['FIBR_PREDS', 'FIBR_JELUD', 'JELUD_TAH', 'PREDS_TAH', 'A_V_BLOK']].max(axis=1, skipna=True).sum()
new_group_arrhythmia_sum = y_train_meta['Severe_Arrhythmias_And_Blocks'].sum()

if original_arrhythmia_sum == new_group_arrhythmia_sum:
    print(f"✅ Arrhythmia & Blocks Grouping Passed: Successfully mapped {int(new_group_arrhythmia_sum)} patients into the group.")
else:
    print(f"❌ Warning: Mathematical gap in arrhythmia grouping. Original Max Sum: {original_arrhythmia_sum} vs New Group Sum: {new_group_arrhythmia_sum}")

# Check 3: Verify the binary lethal outcome tracking (Overall Mortality)
original_deaths = (y_train['LET_IS'] > 0).sum()
new_deaths = y_train_meta['Overall_Mortality'].sum()

if original_deaths == new_deaths:
    print(f"✅ Mortality Mapping Passed: Original Deaths ({original_deaths}) match New Target Deaths ({int(new_deaths)}).")
else:
    print(f"❌ Warning: Mortality count mismatch. Original: {original_deaths} vs New: {new_deaths}")


=== VERIFYING GROUPING INTEGRITY (TRAIN SET ONLY) ===
✅ Row Alignment Check Passed: Both X and New Y contain exactly 1360 rows.
✅ Arrhythmia & Blocks Grouping Passed: Successfully mapped 247 patients into the group.
✅ Mortality Mapping Passed: Original Deaths (217) match New Target Deaths (217).


In [ ]:
# Save the restructured target matrix directly to your project directory
y_train_meta.to_csv("/content/drive/MyDrive/Mid-term Project/y_train_meta_ready.csv", index=True)
print("💾 File successfully saved to: /content/drive/MyDrive/Mid-term Project/y_train_meta_ready.csv")


💾 File successfully saved to: /content/drive/MyDrive/Mid-term Project/y_train_meta_ready.csv


# Step 7: Split the feautre (X) in to 3 separate admission days

In [ ]:
# Strictly maps features by their clinical meaning, ignoring their file position
# to guarantee zero data leakage.

# --- 📅 CLINICAL GROUP 1: ER ADMISSION (Day 1 Features) ---
# Patient background, medical history, chronic conditions, and early ER admission vitals.
cols_day1_base = [
    "AGE", "SEX", "INF_ANAM", "STENOK_AN", "FK_STENOK", "IBS_POST", "GB",
    "SIM_GIPERT", "DLIT_AG", "ZSN_A", "TIME_B_S", "S_AD_ORIT", "D_AD_ORIT",
    "NOT_NA_KB", "LID_KB"  # Ambulance meds - known immediately upon ER arrival
]

# Dynamically capturing all admission binary ECG markers and localization flags
ecg_and_location_cols = [
    col for col in X_train.columns
    if col.startswith(("ritm_ecg_p_", "n_r_ecg_p_", "n_p_ecg_p_")) or col in ["ant_im", "lat_im", "inf_im", "post_im", "IM_PG_P"]
]
features_day1 = cols_day1_base + ecg_and_location_cols


# --- 📅 CLINICAL GROUP 2: MORNING LABS & ENZYMES (Day 2 Features) ---
# Laboratory results that arrive on the morning of Day 2, and initial treatments.
cols_day2_new = [
    "K_BLOOD", "L_BLOOD", "ROE", "ALT_BLOOD", "AST_BLOOD",
    "fibr_ter_01", "fibr_ter_02", "fibr_ter_03", "fibr_ter_05", "fibr_ter_06", "fibr_ter_07", "fibr_ter_08"
]


# --- 📅 CLINICAL GROUP 3: LATE COMPLICATIONS & INTERVENTIONS (Day 3 Features) ---
# Medications and interventions administered throughout the ICU stay up to Day 3.
cols_day3_new = [
    "NITR_S", "ANT_CA_S", "GEPAR_S", "FIBR_REM_S", "TIOP_S", "LID_S", "B_BLOK_S"
]

# ------------------------------------------------------------------------------
# 🛡️ EXPLICIT CHROMATOGRAPHIC SAFETY ROUTING FOR UNMAPPED COLUMNS
# ------------------------------------------------------------------------------
all_mapped_cols = set(features_day1 + cols_day2_new + cols_day3_new)
remaining_features = [col for col in X_train.columns if col not in all_mapped_cols]

# Safely direct unmapped background columns to Day 1, and ICU interventions to Day 3
for col in remaining_features:
    # Most unmapped columns in this UCI dataset are baseline patient history / comorbidities
    # like 'sah_diab' (Diabetes) or 'obes' (Obesity). These belong to Day 1.
    if any(keyword in col.lower() for keyword in ['_an', 'diab', 'obes', 'post_im', 'hvd', 'nk', 'scand']):
        features_day1.append(col)
    else:
        # Late ICU features or metrics go safely to Day 3
        cols_day3_new.append(col)

# ------------------------------------------------------------------------------
# ⏳ CONSTRUCTING THE FINAL CUMULATIVE TIME-SEGMENTED DATAFRAMES
# ------------------------------------------------------------------------------
cols_d1_final = [c for c in features_day1 if c in X_train.columns]
cols_d2_final = cols_d1_final + [c for c in cols_day2_new if c in X_train.columns]
cols_d3_final = cols_d2_final + [c for c in cols_day3_new if c in X_train.columns]

# Generating clean, independent dataframes split into 3 distinct operational windows
X_train_d1 = X_train[cols_d1_final].copy()
X_train_d2 = X_train[cols_d2_final].copy()
X_train_d3 = X_train[cols_d3_final].copy()

print("=== 🎉 STEP 4 SUCCESSFUL: DATA SEGMENTED INTO 3 ADMISSION DAYS ===")
print(f"Day 1 Slice Shape: {X_train_d1.shape} (Comprehensive Baseline Admissions)")
print(f"Day 2 Slice Shape: {X_train_d2.shape} (Cumulative Vitals + Labs)")
print(f"Day 3 Slice Shape: {X_train_d3.shape} (Full Clinical Dataset)")


=== 🎉 STEP 4 SUCCESSFUL: DATA SEGMENTED INTO 3 ADMISSION DAYS ===
Day 1 Slice Shape: (1360, 46) (Comprehensive Baseline Admissions)
Day 2 Slice Shape: (1360, 58) (Cumulative Vitals + Labs)
Day 3 Slice Shape: (1360, 107) (Full Clinical Dataset)


In [ ]:
X_train_d1.dtypes
X_train_d2.dtypes
X_train_d3.dtypes

,0
AGE,float64
SEX,int64
INF_ANAM,float64
STENOK_AN,float64
FK_STENOK,float64
...,...
ANT_CA_S_n,float64
GEPAR_S_n,float64
ASP_S_n,float64
TIKL_S_n,float64


# Save raw sliced files

In [ ]:
# ==============================================================================
# 💾 EXPORTING RAW TIMELINE DATASETS TO GOOGLE DRIVE
# ==============================================================================
folder_drive = "/content/drive/MyDrive/Mid-term Project/"
os.makedirs(folder_drive, exist_ok=True)

print(f"\n⏳ Serializing and exporting raw sliced CSV files to: {folder_drive}")

# CHANGED index=True to guarantee patient alignment across features and targets
X_train_d1.to_csv(os.path.join(folder_drive, "X_train_day1_raw_sliced.csv"), index=True)
X_train_d2.to_csv(os.path.join(folder_drive, "X_train_day2_raw_sliced.csv"), index=True)
X_train_d3.to_csv(os.path.join(folder_drive, "X_train_day3_raw_sliced.csv"), index=True)

print("\n=== 💾 RAW SLICED CSV FILES SUCCESSFULLY SAVED TO DRIVE! ===")



⏳ Serializing and exporting raw sliced CSV files to: /content/drive/MyDrive/Mid-term Project/

=== 💾 RAW SLICED CSV FILES SUCCESSFULLY SAVED TO DRIVE! ===


# Step 8: Imputation, encoding and scaling separately for each admission day

In [ ]:
def fit_and_process_day_data(X_day):
    """
    Fits preprocessing objects on training data and transforms it,
    while preserving strict index alignment.
    """
    X_day = X_day.copy()
    fitted_objects = {}  # Dictionary to store trained transformers for the Test set

    # 1. Automatic classification based on data types and unique value thresholds
    numeric_cols = [c for c in X_day.columns if X_day[c].dtype in ['float64', 'int64'] and X_day[c].nunique() > 10]
    categorical_cols = [c for c in X_day.columns if c not in numeric_cols]

    fitted_objects['numeric_cols'] = numeric_cols
    fitted_objects['categorical_cols'] = categorical_cols

    # 2. Handle missing values (Imputation)
    num_imputer = SimpleImputer(strategy='median')
    if numeric_cols:
        X_day[numeric_cols] = num_imputer.fit_transform(X_day[numeric_cols])
    fitted_objects['num_imputer'] = num_imputer

    cat_imputer = SimpleImputer(strategy='most_frequent')
    if categorical_cols:
        X_day[categorical_cols] = cat_imputer.fit_transform(X_day[categorical_cols])
    fitted_objects['cat_imputer'] = cat_imputer

    # 3. Apply One-Hot Encoding only to true multi-class variables (>2 distinct categories)
    true_cat_cols = [c for c in categorical_cols if X_day[c].nunique() > 2]
    fitted_objects['true_cat_cols'] = true_cat_cols

    if true_cat_cols:
        encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        encoded_array = encoder.fit_transform(X_day[true_cat_cols])

        # CRITICAL FIX: Match the encoded dataframe index PERFECTLY with X_day index
        encoded_df = pd.DataFrame(
            encoded_array,
            columns=encoder.get_feature_names_out(true_cat_cols),
            index=X_day.index
        )
        # Drop old categorical columns WITHOUT reset_index()
        X_day = X_day.drop(columns=true_cat_cols)
        X_day = pd.concat([X_day, encoded_df], axis=1)
        fitted_objects['encoder'] = encoder

    # 4. Normalize continuous features using StandardScaler
    scaler = StandardScaler()
    if numeric_cols:
        X_day[numeric_cols] = scaler.fit_transform(X_day[numeric_cols])
    fitted_objects['scaler'] = scaler

    return X_day, fitted_objects


print("⏳ Processing, imputing, and scaling features for each isolated day...")

# Transforming each day's dataframe and saving the fitted transformers
X_train_d1_ready, transformers_d1 = fit_and_process_day_data(X_train_d1)
X_train_d2_ready, transformers_d2 = fit_and_process_day_data(X_train_d2)
X_train_d3_ready, transformers_d3 = fit_and_process_day_data(X_train_d3)

print("\n=== 🎉 Preprocessing and Scaling Completed Successfully! ===")
print(f"Processed Day 1 Shape: {X_train_d1_ready.shape} (Patient Index Preserved)")
print(f"Processed Day 2 Shape: {X_train_d2_ready.shape} (Patient Index Preserved)")
print(f"Processed Day 3 Shape: {X_train_d3_ready.shape} (Patient Index Preserved)")


⏳ Processing, imputing, and scaling features for each isolated day...

=== 🎉 Preprocessing and Scaling Completed Successfully! ===
Processed Day 1 Shape: (1360, 99) (Patient Index Preserved)
Processed Day 2 Shape: (1360, 111) (Patient Index Preserved)
Processed Day 3 Shape: (1360, 187) (Patient Index Preserved)


In [ ]:
X_train_d1_ready.dtypes
X_train_d2_ready.dtypes
X_train_d3_ready.dtypes

,0
AGE,float64
SEX,float64
SIM_GIPERT,float64
S_AD_ORIT,float64
D_AD_ORIT,float64
...,...
NOT_NA_2_n_2.0,float64
NOT_NA_2_n_3.0,float64
NOT_NA_3_n_0.0,float64
NOT_NA_3_n_1.0,float64


# Step 9: Quality control- Santiy check

In [ ]:
# 1. Counting missing values (Should be 0)
nan_day1 = X_train_d1_ready.isnull().sum().sum()
nan_day2 = X_train_d2_ready.isnull().sum().sum()
nan_day3 = X_train_d3_ready.isnull().sum().sum()

# 2. Extracting row counts (Should all be exactly 1360)
rows_d1 = X_train_d1_ready.shape[0]
rows_d2 = X_train_d2_ready.shape[0]
rows_d3 = X_train_d3_ready.shape[0]
rows_y = y_train_meta.shape[0]  # CHANGED HERE to check your consolidated meta-targets!

print("=== 📋 Preprocessing Quality Control Report ===")
print(f"Missing Values left (Day 1 / Day 2 / Day 3): {nan_day1} / {nan_day2} / {nan_day3}")
print(f"Patient Counts (Day 1 / Day 2 / Day 3 / Meta-Targets): {rows_d1} / {rows_d2} / {rows_d3} / {rows_y}")

# Validation logic: Checking for zero NaNs AND verifying that all row counts match 1360
is_nan_clean = (nan_day1 == 0 and nan_day2 == 0 and nan_day3 == 0)
are_rows_intact = (rows_d1 == 1360 and rows_d2 == 1360 and rows_d3 == 1360 and rows_y == 1360)

is_data_clean = is_nan_clean and are_rows_intact

if is_data_clean:
    print("\n🎉 Sanity Check Passed! Data size is 100% intact (1360 patients) and completely imputed.")
else:
    print("\n⚠️ Sanity Check Failed!")
    if not is_nan_clean:
        print("- Missing values (NaNs) were detected.")
    if not are_rows_intact:
        print(f"- Data loss detected! Expected 1360 rows, but got sizes: {rows_d1}, {rows_d2}, {rows_d3}, {rows_y}")
    print("Please review your preprocessing pipeline before attempting to save.")


=== 📋 Preprocessing Quality Control Report ===
Missing Values left (Day 1 / Day 2 / Day 3): 0 / 0 / 0
Patient Counts (Day 1 / Day 2 / Day 3 / Meta-Targets): 1360 / 1360 / 1360 / 1360

🎉 Sanity Check Passed! Data size is 100% intact (1360 patients) and completely imputed.


# Save the ready files

In [ ]:
# Define the project directory on Google Drive
folder_drive = "/content/drive/MyDrive/Mid-term Project/"

print("⏳ Saving final fully processed and scaled datasets (for Baseline Modeling)...")

# Exporting fully cleaned, encoded, and normalized datasets WITH patient index preserved (index=True)
X_train_d1_ready.to_csv(os.path.join(folder_drive, "X_train_day1_ready.csv"), index=True)
X_train_d2_ready.to_csv(os.path.join(folder_drive, "X_train_day2_ready.csv"), index=True)
X_train_d3_ready.to_csv(os.path.join(folder_drive, "X_train_day3_ready.csv"), index=True)

# Exporting the synchronized consolidated 5 target categories (y_train_meta) instead of the old y_train
y_train_meta.to_csv(os.path.join(folder_drive, "y_train_ready.csv"), index=True)

print("\n=== 💾 ALL BASELINE TRAIN DATASETS SUCCESSFULLY SERIALIZED TO DRIVE! ===")


⏳ Saving final fully processed and scaled datasets (for Baseline Modeling)...

=== 💾 ALL BASELINE TRAIN DATASETS SUCCESSFULLY SERIALIZED TO DRIVE! ===


## 🏁 Preprocessing Summary & Next Step

### 📊 What Was Done in Preprocessing?
* **Initial Cleaning**: Removed highly incomplete columns such as (`KFK_BLOOD`, `IBS_NASL`).
* **Target Refinement**: Converted the lethal outcome column (`LET_IS`) into a strict binary format (0 for Survived, 1 for Deceased) to match the other 11 complications.
* **Target Consolidation**: Merged the 12 original multi-class complications into **5 clinically distinct, binary target categories** (including overall mortality), restructuring the project into a robust Multi-label Classification framework.
* **Temporal Segmentation**: Segmented the feature space (`X_train`) chronologically into 3 distinct operational windows (Day 1, Day 2, and Day 3).
* **Imputation & Encoding**: Handled missing values (Median for continuous, Mode for categorical variables) and applied One-Hot Encoding to eliminate numerical ranking bias.
* **Feature Scaling**: Standardized all continuous variables (like Age and ER Blood Pressure) to an identical scale to guarantee balanced model training.

---

### 💾 Generated CSV Files Saved to Google Drive:
All files have been explicitly exported with `index=True` to secure structural alignment across feature and target states.

* **Raw Chronological Splits (For Feature Engineering):**
  * `X_train_day1_raw_sliced.csv`
  * `X_train_day2_raw_sliced.csv`
  * `X_train_day3_raw_sliced.csv`
* **Processed Slices (For Immediate Baseline Modeling):**
  * `X_train_day1_ready.csv` (99 expanded features)
  * `X_train_day2_ready.csv` (111 expanded features)
  * `X_train_day3_ready.csv` (187 expanded features)
* **Consolidated Multi-Label Targets:**
  * `y_train_ready.csv` (The final 5-column clinical outcome target matrix)

---


### 🚀 What Is the Next Step?
**Feature Engineering Notebook 3**

